# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset titled:
**Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review**, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined by a Croissant JSON-LD schema, accessible at:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Install mlcroissant if missing
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL 
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)

## 2. Data Overview
Review available record sets, fields, and their `@id`s, as defined by the Croissant schema.

You can list information about each record set and its fields to understand what tabular data is available.

In [ ]:
# List record sets and preview their fields
record_sets = dataset.record_sets()

print('Available record sets:')
for rs in record_sets:
    print(f'- RecordSet: {rs.id} (name={rs.name})')
    fields = rs.fields
    print('  Fields:')
    for field in fields:
        print(f'    • {field.id} (name={field.name}, type={field.data_type})')
    print('  ---')

## 3. Data Extraction
Load data from each record set into DataFrames. Use the record set and field `@id`s as shown above to reference entities directly.

We will extract all available record sets as DataFrames and print their columns using field `@id`.

In [ ]:
# Extract all record sets into pandas DataFrames using their @id
dataframes = {}

for rs in dataset.record_sets():
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f'Columns for RecordSet {rs.id}:')
    print(df.columns.tolist())
    print(df.head(), '\n---\n')

# Choose one record set for detailed EDA (take the first found)
main_record_set_id = next(iter(dataframes))
main_df = dataframes[main_record_set_id]
print(f'First record set selected: {main_record_set_id}')

## 4. Exploratory Data Analysis (EDA)
We will apply several common steps:
- Filtering numeric fields, e.g., age, hormone levels
- Normalizing numeric fields
- Grouping by a categorical field, if available

All fields or columns referenced by their `@id`. For the demonstration, we will attempt to use the first numeric and first categorical field found.

In [ ]:
# Find numeric and categorical field @ids from the record set
rs_obj = None
for rs in dataset.record_sets():
    if rs.id == main_record_set_id:
        rs_obj = rs
        break

numeric_field_id = None
group_field_id = None

for field in rs_obj.fields:
    if field.data_type in ['Float', 'Integer', 'Number'] and numeric_field_id is None:
        numeric_field_id = field.id
    if field.data_type == 'Text' and group_field_id is None:
        group_field_id = field.id

print(f'Numeric field selected: {numeric_field_id}')
print(f'Group (categorical) field selected: {group_field_id}')

threshold = 10
if numeric_field_id in main_df.columns:
    filtered_df = main_df[main_df[numeric_field_id].apply(pd.to_numeric, errors='coerce') > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[numeric_field_id + '_normalized'] = (
        filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce') -
        filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce').mean()
    ) / filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce').std()

    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

    if group_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print('No suitable numeric field found for EDA.')

## 5. Visualization
Visualize numeric distributions and relationships between fields referenced by their `@id`s, using matplotlib or pandas plotting.

In [ ]:
# Visualize filtered numeric data
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(6,4))
    filtered_df[numeric_field_id].apply(pd.to_numeric, errors='coerce').hist(bins=10)
    plt.title(f'Distribution of {numeric_field_id} (filtered > {threshold})')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id in main_df.columns:
        plt.figure(figsize=(7,4))
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print('No numeric field for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and process the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities via their `@id`. We extracted the available record sets, explored numeric and categorical fields, filtered and normalized data, and visualized distributions.

This serves as a reproducible template for the FAIR exploration of tabular datasets modeled with Croissant and enables further domain-specific analyses. Please refer to the dataset metadata and Croissant schema for additional details about columns, structure, and context.